# 03 — Partitioning, QUBO, and QAOA

Week 2: H/O/S partitioning with tunable dials, QUBO formulation for the O-set, and a QAOA prototype validated against exact brute-force enumeration.

In [ ]:
!git clone https://github.com/YOUR_USERNAME/vanguard-quantum-portfolio.git
%cd vanguard-quantum-portfolio
!pip install -e ".[quantum]" -q

## Scoring

Four named dials -- growth, income, drawdown, cost -- combined from z-scored components.

In [ ]:
from vqportfolio.market_data.loader import load_prices
from vqportfolio.market_data.overlays import compute_returns_and_risk, synthetic_cost_and_yield
from vqportfolio.partitioning.scoring import compute_asset_scores, per_asset_max_drawdown, Dials
from vqportfolio.config import TICKERS

prices, used_synthetic = load_prices()
assert not used_synthetic, 'Running on synthetic fallback -- check internet access.'
mu, sigma, log_returns = compute_returns_and_risk(prices)
overlay = synthetic_cost_and_yield(TICKERS, log_returns)
mdd = per_asset_max_drawdown(prices)

dials = Dials()  # try Dials(income=1.5, cost=1.5) to see the ranking shift
scores_df = compute_asset_scores(mu, sigma, overlay['cost_bps'], overlay['yield'], mdd, dials)
scores_df.round(3)

## Partition into H / O / S

In [ ]:
from vqportfolio.partitioning.partition import partition_assets, build_locked_allocation, PartitionConfig

pconfig = PartitionConfig()
partition = partition_assets(scores_df['score'], pconfig)
h_weights, o_budget = build_locked_allocation(scores_df['score'], partition, pconfig)

print('H:', partition['H'])
print('O:', partition['O'])
print('S:', partition['S'])
print()
h_weights.round(4)

## QUBO for the O-set, then QAOA vs. exact

In [ ]:
from vqportfolio.config import ASSET_CLASS_OF, ASSET_CLASS_CAPS
from vqportfolio.quantum.qubo import build_o_set_qubo
from vqportfolio.quantum.qaoa_solver import solve_with_qaoa_and_validate

class_headroom = {}
for asset_class, cap in ASSET_CLASS_CAPS.items():
    idx = [t for t in partition['H'] if ASSET_CLASS_OF[t] == asset_class]
    used = h_weights.loc[idx].sum() if idx else 0.0
    class_headroom[asset_class] = max(cap - used, 0.0)

qubo_result = build_o_set_qubo(partition['O'], mu, sigma, overlay['cost_bps'], o_budget, class_headroom)
comparison = solve_with_qaoa_and_validate(qubo_result, o_budget, class_headroom)

print('QAOA matches exact optimum:', comparison.qaoa_matches_exact)
print('Objective gap:', abs(comparison.qaoa_objective - comparison.exact_objective))
comparison.qaoa_weights.round(4)

## Full pipeline vs. classical Markowitz (single-instance check, not a rigor claim)

In [ ]:
from vqportfolio.pipeline import run_pipeline, portfolio_stats

result, used_synthetic, mu, sigma = run_pipeline()
hos_stats = portfolio_stats(result.qaoa_full_weights, mu, sigma)
mw = result.markowitz_result

print('H/O/S + QAOA  return:', hos_stats['expected_return'], ' risk:', hos_stats['risk_variance'])
print('Markowitz     return:', mw['expected_return'], ' risk:', mw['risk_variance'])

## Sanity checks

In [ ]:
!python -m tests.test_week2_sanity